# Setup

In [1]:
import os

import gymnasium as gym
import pandas as pd
from skopt.space import Real
import time

from popy.simulation_tools import *
from popy.io_tools import load_behavior
from popy.behavior_data_tools import *
from popy.simulation_helpers import fit_simulate, fit_agent, fit_agent_graddesc
from popy.config import PROJECT_PATH_LOCAL

/home/uzsombi/.config/matplotlib is not a writable directory
Matplotlib created a temporary cache directory at /tmp/matplotlib-ovwz7zzw because there was an issue with the default path (/home/uzsombi/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Matplotlib is building the font cache; this may take a moment.


## Init

In [2]:
MODELS = {
    "Foraging": {
        "agent_class": ForagingAgent,
        "fixed_params": {"reset_on_switch": True},
        "free_params": ["alpha", "beta", "V0"],
    },
    "Inferential RL - stickiness": {
        "agent_class": QLearner,
        "fixed_params": {"structure_aware": True},
        "free_params": ["alpha", "beta", "stickiness_bias"],
    },
    "Standard RL - stickiness": {
        "agent_class": QLearner,
        "fixed_params": {"structure_aware": False},
        "free_params": ["alpha", "beta", "stickiness_bias"],
    },
}

fit_params = {
    "epsilon": Real(0.01, 1, name='epsilon'),
    "alpha": Real(0.01, 0.9, name="alpha"),
    "beta": Real(0.05, 50.0, name="beta"),
    "stickiness_bias": Real(0.0, 10.0, name="stickiness_bias"),
    "V0": Real(0.05, 0.7, name="V0"),
}

gp_params = {
    "n_calls": 3,
    "n_initial_points": 2,
    "n_jobs": 1,
    "verbose": False,
}

n_bootstrap = 5

In [3]:
def bootstrap_sessions(behav, bootstrap_idx=None):
    """
    Bootstrap sessions from the original data with unique IDs for each copy.
    
    Each sampled session gets a unique ID: 
    - "session_id_copy0", "session_id_copy1", etc. (or with bootstrap_idx if provided)
    """
    session_ids = behav["session"].unique()
    bootstrapped_sessions = np.random.choice(session_ids, size=len(session_ids), replace=True)
    
    # Track how many times each session has been selected
    session_count = {}
    data_chunks = []
    
    for orig_sid in bootstrapped_sessions:
        if orig_sid not in session_count:
            session_count[orig_sid] = 0
        else:
            session_count[orig_sid] += 1
        
        # Get the data for this session
        session_data = behav[behav["session"] == orig_sid].copy()
        
        # Create unique ID for this copy
        if bootstrap_idx is not None:
            new_id = f"{orig_sid}_boot{bootstrap_idx}_copy{session_count[orig_sid]}"
        else:
            new_id = f"{orig_sid}_copy{session_count[orig_sid]}"
        
        # Update session ID to the unique identifier
        session_data["session"] = new_id
        data_chunks.append(session_data)
    
    bootstrapped_behav = pd.concat(data_chunks, ignore_index=True)
    return bootstrapped_behav

# Simulate models

## gp_minimize

In [4]:
def run_bootstrap_fitting(
    agent_class,
    param_space,
    env,
    behav_data,
    fixed_params,
    model_name,
    n_bootstrap=10,
    gp_params=None,
    output_csv=None,
):
    """
    Run model fitting N times on bootstrapped data and save results to CSV.
    
    Parameters:
    -----------
    agent_class : class
        Agent class to fit
    param_space : list
        Parameter space for optimization
    env : gym.Env
        Environment
    behav_data : pd.DataFrame
        Original behavioral data
    fixed_params : dict
        Fixed parameters for agent
    model_name : str
        Name of the model (for results tracking)
    n_bootstrap : int
        Number of bootstrap iterations
    gp_params : dict, optional
        Parameters for GP minimize (n_calls, n_initial_points, n_jobs, verbose)
    output_csv : str, optional
        Path to save results CSV. If None, does not save individual CSV (useful for batch processing).
    
    Returns:
    --------
    results_df : pd.DataFrame
        DataFrame with all fitting results
    """
    
    if gp_params is None:
        gp_params = {
            "n_calls": 100,
            "n_initial_points": 50,
            "n_jobs": -1,
            "verbose": False,
        }
    
    results_list = []
    
    print(f"Starting bootstrap fitting for model: {model_name}")
    print(f"Running {n_bootstrap} bootstrap iterations...\n")
    
    for bootstrap_idx in range(n_bootstrap):
        print(f"Bootstrap {bootstrap_idx + 1}/{n_bootstrap}...", end=" ", flush=True)
        
        # Bootstrap the data
        bootstrapped_behav = bootstrap_sessions(behav_data, bootstrap_idx=bootstrap_idx)
        
        # Fit the model
        start_time = time.time()
        result = fit_agent(
            agent_class=agent_class,
            param_space=param_space,
            env=env,
            behav_data=bootstrapped_behav,
            fixed_params=fixed_params,
            fit_on="ll",
            n_calls=gp_params["n_calls"],
            n_initial_points=gp_params["n_initial_points"],
            n_jobs=gp_params["n_jobs"],
            verbose=gp_params["verbose"],
        )
        elapsed_time = time.time() - start_time
        
        # Flatten the results dictionary
        row = {
            "model": model_name,
            "bootstrap_idx": bootstrap_idx,
        }
        
        # Add parameters (unpacked from best_params dict)
        if isinstance(result["best_params"], dict):
            for param_name, param_value in result["best_params"].items():
                row[param_name] = param_value
        
        # Add results metrics
        row["best_ll"] = result["best_ll"]
        row["bic"] = result["bic"]
        row["lpt"] = result.get("lpt", None)
        row["fit_time_sec"] = elapsed_time
        
        results_list.append(row)
        
        print(f"LL={result['best_ll']:.4f}, BIC={result['bic']:.4f}, Time={elapsed_time:.2f}s")
    
    # Create DataFrame
    results_df = pd.DataFrame(results_list)
    
    # Optionally save to CSV
    if output_csv is not None:
        results_df.to_csv(output_csv, index=False)
        print(f"\n✓ Results saved to: {output_csv}")
        print(f"\nSummary statistics:")
        print(results_df[["best_ll", "bic", "lpt", "fit_time_sec"]].describe())
    
    return results_df

In [ ]:
# Run bootstrap fitting for both monkeys
for monkey in ["ka", "po"]:
    print(f"\n\n{'#'*70}")
    print(f"# Processing Monkey: {monkey.upper()}")
    print(f"{'#'*70}\n")
    
    # Load data for this monkey
    env = gym.make("zsombi/monkey-bandit-task-v0", n_arms=3, max_episode_steps=100_000)
    behav_monkey = load_behavior(monkey)
    behav_monkey = drop_time_fields(behav_monkey)
    behav_monkey = add_switch_info(behav_monkey)
    behav_monkey = convert_column_format(behav_monkey, original='behavior')
    behav_monkey = behav_monkey.dropna()
    
    print(f"Loaded {len(behav_monkey)} trials from monkey {monkey}\n")
    
    # Run bootstrap fitting for all models
    all_results = []
    
    output_csv = os.path.join(
        PROJECT_PATH_LOCAL, 
        "notebooks", 
        "behav_modeling", 
        "fitting",
        "results", 
        "bootstrap", 
        f"{monkey}_bootstrap_all_models.csv"
    )
    
    # Create output directory if it doesn't exist
    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    
    for model_name, model_config in MODELS.items():
        print(f"\n{'='*60}")
        print(f"Running bootstrap for: {model_name}")
        print(f"{'='*60}")
        
        agent_class = model_config["agent_class"]
        fixed_params = model_config["fixed_params"]
        free_params = model_config["free_params"]
        param_space = [fit_params[param] for param in free_params]
        
        # Run bootstrap fitting (returns dataframe, we'll append to list)
        results_df = run_bootstrap_fitting(
            agent_class=agent_class,
            param_space=param_space,
            env=env,
            behav_data=behav_monkey,
            fixed_params=fixed_params,
            model_name=model_name,
            n_bootstrap=10,  # Adjust as needed
            gp_params=gp_params,
            output_csv=None,  # Don't save individual CSVs
        )
        
        all_results.append(results_df)
    
    # Combine all results and save to single CSV
    combined_results = pd.concat(all_results, ignore_index=True)
    combined_results.to_csv(output_csv, index=False)
    
    print(f"\n{'='*60}")
    print(f"✓ Monkey {monkey.upper()}: All results saved to: {output_csv}")
    print(f"{'='*60}")
    print(f"\nCombined summary statistics:")
    print(combined_results.groupby("model")[["best_ll", "bic", "lpt", "fit_time_sec"]].describe())

print(f"\n\n{'#'*70}")
print(f"# ✓ All monkeys processed successfully!")
print(f"{'#'*70}")



######################################################################
# Processing Monkey: KA
######################################################################

Loaded 23860 trials from monkey ka


Running bootstrap for: Foraging
Starting bootstrap fitting for model: Foraging
Running 10 bootstrap iterations...

Bootstrap 1/10... LL=-6437.3205, BIC=12904.9224, Time=29.59s
Bootstrap 2/10... LL=-18679.2565, BIC=37388.7673, Time=25.47s
Bootstrap 3/10... LL=-9026.2290, BIC=18082.7682, Time=18.58s
Bootstrap 4/10... LL=-5890.0489, BIC=11810.0659, Time=23.90s
Bootstrap 5/10... LL=-20817.1125, BIC=41664.4828, Time=30.79s
Bootstrap 6/10... LL=-7137.5420, BIC=14305.3252, Time=29.26s
Bootstrap 7/10... LL=-8743.5300, BIC=17517.2741, Time=22.00s
Bootstrap 8/10... LL=-8286.8057, BIC=16603.8229, Time=26.20s
Bootstrap 9/10... LL=-22279.6416, BIC=44589.3897, Time=25.75s
Bootstrap 10/10... LL=-11946.2155, BIC=23922.6385, Time=23.73s

Running bootstrap for: Inferential RL - stickiness
Starting boo

: 